# Notebook 18 — Forward Surrogate: 1D-CNN Decoder (A2)

**Goal**: Train an A2-type forward surrogate that predicts a 1000-point absorption
spectrum from 20 design parameters using **1D transposed convolutions** (decoding).

| Aspect | Detail |
|--------|--------|
| **Architecture** | MLP stem (20→256→128×32) + 4× ConvTranspose1d → 1000-point spectrum |
| **Input** | 20 normalised design parameters |
| **Output** | 1000-point absorption spectrum |
| **Data** | 1M LHS samples, 80/10/10 split |

In [ ]:
# ============================================================
# Cell 1 — Imports & Configuration
# ============================================================
import sys, os, time, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, os.path.abspath('../src'))
from Theoretical_model import calculate_acoustic_properties
from physics_guided_CD_FiLM import PARAM_RANGES, validate_and_clip_parameters

assert torch.cuda.is_available(), 'CUDA required'
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
print(f'Device: {torch.cuda.get_device_name()}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths
DATA_PATH  = '../data/lhs_data_full_spectrum.npz'
MODEL_DIR  = '../models'
SAVE_PATH  = os.path.join(MODEL_DIR, 'forward_surrogate_cnn.pth')
SAVE_SCALER = os.path.join(MODEL_DIR, 'forward_surrogate_cnn_scaler.pkl')

# Hyper-parameters
BATCH_SIZE = 1024
EPOCHS     = 50
LR_INIT    = 1e-3
LR_MIN     = 1e-5
NUM_PARAMS = 20
NUM_FREQ   = 1000

PARAM_NAMES = ['d1','d2','d3','d4','d5','d6','d7','d8','d9','d10',
               'm2','m3','m5','m6','m8','m9','rho','eta','E','nu']

print('\n✓ Imports & config ready')

In [ ]:
# ============================================================
# Cell 2 — Load Data & Split
# ============================================================
print('Loading NPZ …')
t0 = time.time()
raw = np.load(DATA_PATH)
params_all  = raw['params'].astype(np.float32)
spectra_all = raw['spectra'].astype(np.float32)
frequencies = raw['frequencies']
print(f'  Loaded in {time.time()-t0:.1f}s')

X_train, X_temp, Y_train, Y_temp = train_test_split(
    params_all, spectra_all, test_size=0.2, random_state=SEED)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.5, random_state=SEED)

print(f'  Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')

scaler_x = MinMaxScaler()
X_train_n = scaler_x.fit_transform(X_train).astype(np.float32)
X_val_n   = scaler_x.transform(X_val).astype(np.float32)
X_test_n  = scaler_x.transform(X_test).astype(np.float32)

def make_loader(params_n, spec, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(params_n, dtype=torch.float32),
                       torch.tensor(spec, dtype=torch.float32))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=0, pin_memory=True)

train_loader = make_loader(X_train_n, Y_train, BATCH_SIZE)
val_loader   = make_loader(X_val_n,   Y_val,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test_n,  Y_test,  BATCH_SIZE, shuffle=False)
print('✓ Data ready')

In [ ]:
# ============================================================
# Cell 3 — 1D-CNN Decoder Model
# ============================================================
class ForwardSurrogateCNN(nn.Module):
    """
    A2: MLP stem → reshape → 1D transposed convolutions → 1000 output.
    
    MLP:   20 → 256 → 128*32 = 4096 → reshape (128, 32)
    ConvT: (128, 32) → (64, 64) → (32, 128) → (16, 256) → (8, 512) → (1, 1000)
    """
    def __init__(self, in_dim=20, out_dim=1000):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        
        # MLP stem
        self.stem = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.BatchNorm1d(256), nn.LeakyReLU(0.01, inplace=True),
            nn.Linear(256, 128 * 32),
            nn.BatchNorm1d(128 * 32), nn.LeakyReLU(0.01, inplace=True),
        )
        
        # Transposed convolutions: upsample 32 → 64 → 128 → 256 → 512
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=4, stride=2, padding=1),  # → 64
            nn.BatchNorm1d(64), nn.LeakyReLU(0.01, inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=4, stride=2, padding=1),   # → 128
            nn.BatchNorm1d(32), nn.LeakyReLU(0.01, inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=4, stride=2, padding=1),   # → 256
            nn.BatchNorm1d(16), nn.LeakyReLU(0.01, inplace=True),
            nn.ConvTranspose1d(16, 8,  kernel_size=4, stride=2, padding=1),   # → 512
            nn.BatchNorm1d(8), nn.LeakyReLU(0.01, inplace=True),
        )
        
        # Final projection: (8, 512) → flatten → 4096 → 1000
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(8 * 512, 1024),
            nn.LeakyReLU(0.01, inplace=True),
            nn.Linear(1024, out_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        h = self.stem(x)             # (B, 4096)
        h = h.view(-1, 128, 32)      # (B, 128, 32)
        h = self.decoder(h)           # (B, 8, 512)
        return self.head(h)           # (B, 1000)


model = ForwardSurrogateCNN(in_dim=NUM_PARAMS, out_dim=NUM_FREQ).to(device)

# Verify output shape
with torch.no_grad():
    dummy = torch.randn(2, NUM_PARAMS, device=device)
    out = model(dummy)
    assert out.shape == (2, NUM_FREQ), f'Unexpected shape: {out.shape}'

total_p = sum(p.numel() for p in model.parameters())
print(f'✓ ForwardSurrogateCNN — {total_p:,} parameters')
print(f'  Output shape verified: {out.shape}')

In [ ]:
# ============================================================
# Cell 4 — Loss, Optimiser, Scheduler
# ============================================================
def build_freq_weights(num_freq=1000, low_cutoff=400, low_weight=2.0, device='cpu'):
    w = torch.ones(num_freq, device=device)
    w[:low_cutoff] = low_weight
    return w / w.mean()

freq_w = build_freq_weights(device=device)

def weighted_mse(pred, target):
    return ((pred - target)**2 * freq_w).mean()

optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
print('✓ Loss & optimiser configured')

In [ ]:
# ============================================================
# Cell 5 — Training Loop
# ============================================================
train_losses, val_losses = [], []
best_val  = float('inf')
best_state = None

print(f'Training {EPOCHS} epochs …\n')
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for params_b, spec_b in train_loader:
        params_b, spec_b = params_b.to(device), spec_b.to(device)
        pred = model(params_b)
        loss = weighted_mse(pred, spec_b)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        ep_loss += loss.item() * params_b.size(0)

    train_losses.append(ep_loss / len(train_loader.dataset))

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for params_b, spec_b in val_loader:
            params_b, spec_b = params_b.to(device), spec_b.to(device)
            pred = model(params_b)
            v_loss += weighted_mse(pred, spec_b).item() * params_b.size(0)

    val_losses.append(v_loss / len(val_loader.dataset))
    scheduler.step()

    if val_losses[-1] < best_val:
        best_val   = val_losses[-1]
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        tag = ' ★'
    else:
        tag = ''

    if epoch % 5 == 0 or epoch == 1:
        lr = scheduler.get_last_lr()[0]
        print(f'  Epoch {epoch:3d}/{EPOCHS}  train {train_losses[-1]:.6e}  '
              f'val {val_losses[-1]:.6e}  lr {lr:.2e}{tag}')

elapsed = time.time() - t_start
print(f'\n✓ Training complete in {elapsed/60:.1f} min — best val = {best_val:.6e}')
model.load_state_dict(best_state)
model.to(device)
print('  Best checkpoint restored')

In [ ]:
# ============================================================
# Cell 6 — Learning Curves
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(range(1, EPOCHS+1), train_losses, label='Train')
ax.semilogy(range(1, EPOCHS+1), val_losses,   label='Val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Weighted MSE (log)')
ax.set_title('A2 CNN Surrogate — Learning Curves')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 7 — Test Metrics
# ============================================================
model.eval()
all_mse = []
with torch.no_grad():
    for params_b, spec_b in test_loader:
        params_b, spec_b = params_b.to(device), spec_b.to(device)
        pred = model(params_b)
        mse = ((pred - spec_b)**2).mean(dim=1)
        all_mse.append(mse.cpu().numpy())

mse_arr = np.concatenate(all_mse)
print('═══════════════════════════════════════════')
print('  TEST METRICS — A2 CNN Forward Surrogate')
print('═══════════════════════════════════════════')
print(f'  Mean spectral MSE   : {mse_arr.mean():.6e}')
print(f'  Median spectral MSE : {np.median(mse_arr):.6e}')
print(f'  Mean RMSE           : {np.sqrt(mse_arr).mean():.6e}')
for p in [90, 95, 99]:
    print(f'  {p}th pctile MSE    : {np.percentile(mse_arr, p):.6e}')
print('═══════════════════════════════════════════')

In [ ]:
# ============================================================
# Cell 8 — Per-Frequency MSE
# ============================================================
per_freq_mse = []
model.eval()
with torch.no_grad():
    for params_b, spec_b in test_loader:
        params_b, spec_b = params_b.to(device), spec_b.to(device)
        pred = model(params_b)
        per_freq_mse.append(((pred - spec_b)**2).cpu().numpy())

per_freq_mse = np.concatenate(per_freq_mse, axis=0).mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(frequencies, per_freq_mse)
ax.axvspan(1, 400, alpha=0.1, color='red', label='Low-freq emphasis zone')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('MSE (log)')
ax.set_title('A2 CNN Surrogate — Per-Frequency MSE')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 9 — Overlay: 6 Spectra
# ============================================================
np.random.seed(77)
show_idx = np.random.choice(len(X_test), 6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
model.eval()
for ax, idx in zip(axes.flat, show_idx):
    params_t = torch.tensor(X_test_n[idx], dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        pred = model(params_t).cpu().numpy().flatten()
    target = Y_test[idx]
    mse = np.mean((pred - target)**2)

    ax.plot(frequencies, target, 'b-',  lw=1.2, label='True (TMM)')
    ax.plot(frequencies, pred,   'r--', lw=1.0, label='A2 CNN')
    ax.set_title(f'MSE={mse:.2e}', fontsize=10)
    ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.supxlabel('Frequency (Hz)')
fig.supylabel('Absorption Coefficient')
fig.suptitle('A2 CNN Surrogate — Sample Predictions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 10 — TMM Cross-Validation (10 samples)
# ============================================================
NUM_TMM = 10
np.random.seed(99)
tmm_idx = np.random.choice(len(X_test), NUM_TMM, replace=False)

tmm_results = []
for idx in tmm_idx:
    params_raw = X_test[idx]
    param_dict = {
        'rho': float(params_raw[16]), 'eta': float(params_raw[17]),
        'E': float(params_raw[18]), 'nu': float(params_raw[19]), 'W': 2000.0,
        'd': [float(params_raw[i]) for i in range(10)],
        'm': {2: float(params_raw[10]), 3: float(params_raw[11]),
              5: float(params_raw[12]), 6: float(params_raw[13]),
              8: float(params_raw[14]), 9: float(params_raw[15])}
    }
    _, alpha_tmm, _, _ = calculate_acoustic_properties(param_dict)

    params_t = torch.tensor(X_test_n[idx], dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        pred = model(params_t).cpu().numpy().flatten()

    mse_vs_data = np.mean((pred - Y_test[idx])**2)
    mse_vs_tmm  = np.mean((pred - alpha_tmm)**2)
    tmm_results.append({'idx': idx, 'mse_data': mse_vs_data, 'mse_tmm': mse_vs_tmm})

print('TMM Cross-Validation (A2 CNN Surrogate):')
for r in tmm_results:
    print(f'  Sample {r["idx"]:6d}  MSE(data)={r["mse_data"]:.4e}  MSE(tmm)={r["mse_tmm"]:.4e}')
print(f'\n  Mean MSE vs TMM: {np.mean([r["mse_tmm"] for r in tmm_results]):.4e}')

In [ ]:
# ============================================================
# Cell 11 — Save Model
# ============================================================
test_metrics = {
    'spectral_mse': float(mse_arr.mean()),
    'spectral_rmse': float(np.sqrt(mse_arr).mean()),
    'spectral_mse_95': float(np.percentile(mse_arr, 95)),
}

torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': {'in_dim': NUM_PARAMS, 'out_dim': NUM_FREQ},
    'test_metrics': test_metrics,
    'train_loss': train_losses,
    'val_loss': val_losses,
    'best_val_loss': best_val,
}, SAVE_PATH)

with open(SAVE_SCALER, 'wb') as f:
    pickle.dump(scaler_x, f)

print(f'✓ Model saved → {SAVE_PATH}  ({os.path.getsize(SAVE_PATH)/1e6:.1f} MB)')
print(f'  Scaler → {SAVE_SCALER}')

In [ ]:
# ============================================================
# Cell 12 — Reload & Verify
# ============================================================
ckpt = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model2 = ForwardSurrogateCNN(**ckpt['architecture']).to(device)
model2.load_state_dict(ckpt['model_state_dict'])
model2.eval()

params_t = torch.tensor(X_test_n[0], dtype=torch.float32, device=device).unsqueeze(0)
with torch.no_grad():
    p1 = model(params_t)
    p2 = model2(params_t)
assert torch.allclose(p1, p2, atol=1e-6)
print('✓ Reload verified — outputs match')

---
## Summary

| Metric | Value |
|--------|-------|
| Architecture | MLP stem + 4× ConvTranspose1d decoder |
| Input → Output | 20 params → 1000-point spectrum |
| Training data | 800K train, 100K val |

This A2 surrogate will be loaded (frozen) by Notebooks 19–24 for all A2-based inverse methods.